In [0]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

data = [(1, 'Eva', 2450), (2, 'Charlie', 3097), (3, 'Ivy', 1953), (4, 'Bob', 3924), (5, 'Bob', 4746), (6, 'Frank', 3120), (7, 'Frank', 1431), (8, 'Frank', 801), (9, 'Jack', 4388), (10, 'Eva', 3268), (11, 'Ivy', 2794), (12, 'Jack', 4153), (13, 'Alice', 4410), (14, 'Ivy', 2865), (15, 'Frank', 2337), (16, 'Grace', 2575), (17, 'Charlie', 2446), (18, 'Bob', 2462), (19, 'Frank', 682), (20, 'Charlie', 819), (21, 'Eva', 4654), (22, 'Jack', 3449), (23, 'Ivy', 4630), (24, 'Eva', 3654), (25, 'Bob', 2662), (26, 'Frank', 741), (27, 'Ivy', 3688), (28, 'Jack', 4729), (29, 'Alice', 2930), (30, 'Charlie', 1085), (31, 'David', 4096), (32, 'Helen', 523), (33, 'Jack', 868), (34, 'Eva', 3094), (35, 'Charlie', 4403), (36, 'Bob', 1953), (37, 'Frank', 1996), (38, 'Alice', 1930), (39, 'Grace', 4408), (40, 'Alice', 4145), (41, 'David', 2478), (42, 'Charlie', 3403), (43, 'David', 2553), (44, 'Grace', 2456), (45, 'Charlie', 3523), (46, 'Frank', 2434), (47, 'Charlie', 2844), (48, 'Jack', 3830), (49, 'Frank', 1889), (50, 'Grace', 3858), (51, 'Eva', 4345), (52, 'Alice', 1584), (53, 'Charlie', 1359), (54, 'Bob', 3538), (55, 'Charlie', 983), (56, 'Frank', 696), (57, 'Alice', 675), (58, 'Alice', 3153), (59, 'Alice', 1802), (60, 'Eva', 4376)]

df = spark.createDataFrame(data, ["id","name","amount"])


df.write.format("delta").mode("overwrite").save("/Volumes/workspace/default/sample_data/details")

delta_df = spark.read.format("delta").load("/Volumes/workspace/default/sample_data/details")

print("Initial Data")
delta_df.show()

# =========================================
#  Incremental Dataset (for MERGE)
# =========================================

incremental_data = [
    (1, "Alice", 3000),   # update
    (2, "Bob", 4000),     # update
    (61, "NewCust1", 2000),  # insert
    (62, "NewCust2", 3500)   # insert
]

inc_df = spark.createDataFrame(incremental_data, ["id","name","amount"])

# =========================================
# 🔹 TASKS
# =========================================

# 1. INSERT new records

# 2. UPDATE existing records

# 3. DELETE records where amount < 1000

# 4. MERGE incremental dataset into main table
#    WHEN MATCHED THEN UPDATE
#    WHEN NOT MATCHED THEN INSERT

# 5. Add new column (category) in incremental dataset
#    and perform schema evolution

# 6. Use DESCRIBE HISTORY

# 7. Perform time travel and restore previous version

print("Start implementing step by step")

Initial Data
+---+-------+------+
| id|   name|amount|
+---+-------+------+
|  1|    Eva|  2450|
|  2|Charlie|  3097|
|  3|    Ivy|  1953|
|  4|    Bob|  3924|
|  5|    Bob|  4746|
|  6|  Frank|  3120|
|  7|  Frank|  1431|
|  8|  Frank|   801|
|  9|   Jack|  4388|
| 10|    Eva|  3268|
| 11|    Ivy|  2794|
| 12|   Jack|  4153|
| 13|  Alice|  4410|
| 14|    Ivy|  2865|
| 15|  Frank|  2337|
| 16|  Grace|  2575|
| 17|Charlie|  2446|
| 18|    Bob|  2462|
| 19|  Frank|   682|
| 20|Charlie|   819|
+---+-------+------+
only showing top 20 rows
Start implementing step by step


In [0]:
data=[(61,"Manasa",4000)]
columns=["id","name","amount"]
new_record=spark.createDataFrame(data,columns)
new_record.display()

id,name,amount
61,Manasa,4000


In [0]:
new_record.write.format("delta").mode("append").save("/Volumes/workspace/default/sample_data/details")


In [0]:
delta_df.display()

id,name,amount
1,Eva,2450
2,Charlie,3097
3,Ivy,1953
4,Bob,3924
5,Bob,4746
6,Frank,3120
7,Frank,1431
8,Frank,801
9,Jack,4388
10,Eva,3268


In [0]:
from delta.tables import DeltaTable
deltaTab=DeltaTable.forPath(spark,"/Volumes/workspace/default/sample_data/details")
deltaTab.update(
    condition="id=61",
    set={"amount":"6000"}
)


DataFrame[num_affected_rows: bigint]

In [0]:
spark.read.format("delta").load("/Volumes/workspace/default/sample_data/details").display()

id,name,amount
1,Eva,2450
2,Charlie,3097
3,Ivy,1953
4,Bob,3924
5,Bob,4746
6,Frank,3120
7,Frank,1431
8,Frank,801
9,Jack,4388
10,Eva,3268


In [0]:
deltaTab.delete("amount<1000")

DataFrame[num_affected_rows: bigint]

In [0]:
deltaTab.alias("target").merge(
    inc_df.alias("source"),
    "target.id=source.id"
).whenMatchedUpdate(set={
    "name":"source.name",
    "amount":"source.amount"
}).whenNotMatchedInsert(values={
    "id":"source.id",
    "name":"source.name",
    "amount":"source.amount"
}).execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
target=spark.read.format("delta").load("/Volumes/workspace/default/sample_data/details")
target.display()


id,name,amount
3,Ivy,1953
4,Bob,3924
5,Bob,4746
6,Frank,3120
7,Frank,1431
9,Jack,4388
10,Eva,3268
11,Ivy,2794
12,Jack,4153
13,Alice,4410


In [0]:
data=[(63,"Jeswanth",5000,"India")]
columns=["id","name","amount","country"]
new_col=spark.createDataFrame(data,columns)
new_col.display()

id,name,amount,country
63,Jeswanth,5000,India


In [0]:
new_col.write.format("delta").mode("append").option("mergeSchema","true").save("/Volumes/workspace/default/sample_data/details")

In [0]:
target=spark.read.format("delta").load("/Volumes/workspace/default/sample_data/details")
target.display()

id,name,amount,country
3,Ivy,1953,null
4,Bob,3924,null
5,Bob,4746,null
6,Frank,3120,null
7,Frank,1431,null
9,Jack,4388,null
10,Eva,3268,null
11,Ivy,2794,null
12,Jack,4153,null
13,Alice,4410,null


In [0]:
from delta.tables import DeltaTable
deltaTab=DeltaTable.forPath(spark,"/Volumes/workspace/default/sample_data/details")

In [0]:
deltaTab.history().display()

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
15,2026-04-14T13:37:47.000Z,70969276671544,manasapuligadda9@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(924304771462199),88b9fd90-ecc9-41c3-b644-5e304b03e152,0414-132420-5suf219a-v2n,14,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1236)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
14,2026-04-14T13:25:41.000Z,70969276671544,manasapuligadda9@gmail.com,MERGE,"Map(predicate -> [""(id#11699L = id#11702L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(924304771462199),ff60389f-b2d9-41e8-8282-74838e374130,0414-132420-5suf219a-v2n,12,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 4119, numTargetBytesRemoved -> 1029, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 3, executionTimeMs -> 4944, materializeSourceTimeMs -> 532, numTargetRowsInserted -> 1, conflictDetectionTimeMs -> 1310, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 1, scanTimeMs -> 2128, numTargetRowsUpdated -> 3, numOutputRows -> 4, numTargetDeletionVectorsRemoved -> 1, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2109)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
13,2026-04-14T13:25:35.000Z,70969276671544,manasapuligadda9@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(924304771462199),81ef297f-c298-4ccb-82b3-a9a2b6ffbc52,0414-132420-5suf219a-v2n,12,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 2704, p25FileSize -> 1607, numDeletionVectorsRemoved -> 1, minFileSize -> 1607, numAddedFiles -> 1, maxFileSize -> 1607, p75FileSize -> 1607, p50FileSize -> 1607, numAddedBytes -> 1607)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
12,2026-04-14T13:25:32.000Z,70969276671544,manasapuligadda9@gmail.com,DELETE,"Map(predicate -> [""(amount#11512L < 1000)""])",null,List(924304771462199),81ef297f-c298-4ccb-82b3-a9a2b6ffbc52,0414-132420-5suf219a-v2n,11,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2118, numDeletionVectorsUpdated -> 0, numDeletedRows -> 9, scanTimeMs -> 1238, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 847)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
11,2026-04-14T13:25:25.000Z,70969276671544,manasapuligadda9@gmail.com,UPDATE,"Map(predicate -> [""(id#11224L = 61)""])",null,List(924304771462199),b94c4baf-0708-4c8d-ac29-2318862899b2,0414-132420-5suf219a-v2n,10,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1029, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 4724, numDeletionVectorsUpdated -> 0, scanTimeMs -> 2673, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1029, rewriteTimeMs -> 2006)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
10,2026-04-14T13:25:17.000Z,70969276671544,manasapuligadda9@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(924304771462199),291db444-82b2-41e0-bcbb-4dc985139d75,0414-132420-5suf219a-v2n,9,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1029)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
9,2026-04-14T13:25:03.000Z,70969276671544,manasapuligadda9@gmail.com,WRITE

In [0]:
df_old=spark.read.format("delta").option("versionAsOf",0).load("/Volumes/workspace/default/sample_data/details")
df_old.display()

id,name,amount
1,Eva,2450
2,Charlie,3097
3,Ivy,1953
4,Bob,3924
5,Bob,4746
6,Frank,3120
7,Frank,1431
8,Frank,801
9,Jack,4388
10,Eva,3268
